# Template application

In [ ]:
entity_template_file = 'entity.templ.html'
attribute_template_file = 'attribute.templ.html'

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>Invalid information. For testing purposes only!</b></span> ')

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
from lxml import etree
import IPython

# valid xml namespaces and schema for Confluence 6 storage format
# See 
xml_namespaces=[
    'xmlns="http://www.w3.org/1999/xhtml"',
    'xmlns:ac="http://www.atlassian.com/schema/confluence/4/ac/"',
    'xmlns:ri="http://www.atlassian.com/schema/confluence/4/ri/"',
    'xmlns:acxhtml="http://www.atlassian.com/schema/confluence/4/"'
]

def encapsulate_storage_format(xml):
    return '<?xml version="1.0"?><root doc="container to properly encapsulate xml" {} >\n{}\n</root>'.format(
            ' '.join(xml_namespaces), xml)

def beautify_xml(flat_xml):
    try:
        encapsulated = encapsulate_storage_format(flat_xml)
        dom = xml.dom.minidom.parseString(encapsulated)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        log.error('Expat error {}'.format(e))
        raise Exception(e)
    
def validate_storage_format(xml_in_storage_format):
        '''Validate if input xml conforms to Confluence storage format specifications'''
        parser = etree.XMLParser(dtd_validation=False)
        try:
            etree.fromstring(encapsulate_storage_format(xml_in_storage_format), parser)
            return None
        except xml.parsers.expat.ExpatError as e:
            m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
            message = 'Malformed xml ' + flat_xml[:50] + ' ...'
            if m:
                lines = wrapped.splitlines()
                messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
            else:
                message = message + flat_xml[:50] + ' ...'
            return message

beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('all fine!')

## Import the Publisher from Publisher.py
PublisherDev is an extension of the external Publisher in Publisher.py. Stable extensions to Publisher should be moved to Publisher.py.

In [ ]:
import Publisher as cp

class PublisherDev(cp.Publisher):
    '''This class'''
    
    def enable_development(self):
        self.version_comment = 'Update due to development / testing' 


# Load datasource

In [ ]:
import json

data = None
with open('testdata/IM-sample.json', 'r') as source:
     data = json.load(source)

entities = data['entities']
# Print some information on what was loaded
print('Model "{}" contains {} entities:'.format(data['model']['name'], len(entities)))

In [ ]:
publisher = PublisherDev({ 'content': {'entities': {}, 'attributes': {}} }, data, None, None, None)
publisher.enable_development()
list(map(lambda e: (e, publisher.translate(entities[e]['name'])), data['entities']))[:5]

## Load page mappings
This will be done during the scan phase in production mode.

In [ ]:
content = publisher.scan_current_content()
len(content)

In [ ]:
elements = [ 'entities', 'attributes', 'relations']

rnd = -1
for element_type in []:
    for key in data[element_type]:
        entitiy = data[element_type][key]
        name = entitiy.get('name')
        rnd -= 1
        publisher.register_page_id(key, rnd)

In [ ]:
publisher.relation_self('ENTI12753', 'RELA13060')

In [ ]:
publisher.relation_other('ENTI12701', 'RELA13060')

# Sandbox for entity template

In [ ]:
test_entity_name = 'ENTI12701' # Attribut
test_entity_name = 'ENTI12753' # Entität

test_entity = data['entities'][test_entity_name]
test_entity_title = publisher.translate(test_entity['name'])
log.warning('Working with entity {} Name: "{}"'.format(test_entity_name, test_entity_title))
test_entity

In [ ]:
entity_template = None
with open('./templates/' + entity_template_file, 'r') as f:
    entity_template = f.read()

assert entity_template

IPython.display.Code(entity_template)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
env.globals.update({ 'util': publisher, 'data': data, 'header': header })

entity_template = env.get_template(entity_template_file)
rendered_entity_template = entity_template.render(key=test_entity_name, item=test_entity)
entity_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_entity_template) # strip comment lines
IPython.display.Code(entity_content_xml)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(test_entity_title, entity_content_xml))

In [ ]:
validate_storage_format(entity_content_xml)

# Sandbox for attribute template

In [ ]:
first_attribute_name = 'ATTR12612'
test_attribute = data['attributes'][first_attribute_name]
assert test_attribute

test_attribute_title = publisher.translate(test_attribute['name'])

log.warning('Working with attribute ' + first_attribute_name + ". Name: " + test_attribute['name']['de'])
test_attribute

In [ ]:
attribute_template = env.get_template(attribute_template_file)
rendered_attribute_template = attribute_template.render(key=first_attribute_name, item=test_attribute, util=publisher, header=header)
attribute_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_attribute_template) # strip comment lines
IPython.display.Code(attribute_content_xml)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(test_attribute_title, attribute_content_xml))

# Publish to test space to verify result in Confluence

In [ ]:
import yaml
import copy

with open('private.yaml') as f:
    config = yaml.safe_load(f)

space_key = config['confluence']['space']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])

In [ ]:
entity_result = confluence.update_or_create(root_page_id, '{} - Test entity'.format(test_entity_title), entity_content_xml, minor_edit=False, version_comment='development and testing')
attribute_result = confluence.update_or_create(root_page_id, '{} - Test attribute'.format(test_attribute_title), attribute_content_xml, minor_edit=True, version_comment='development and testing')

In [ ]:
e_page = entity_result['_links']['webui']
e_uri = '{}{}'.format(config['confluence']['apiurl'], e_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    e_uri, test_entity_title))

In [ ]:
a_page = attribute_result['_links']['webui']
a_uri = '{}{}'.format(config['confluence']['apiurl'], a_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    a_uri, test_attribute_title))